In [ ]:
import datasets

# load cifar-100 dataset
dataset = datasets.load_dataset("cifar100")

In [ ]:
import datasets
from io import BytesIO
from PIL import Image

# 1) Create a tiny mock image parquet locally
def make_png_bytes(color, size=(32, 32)):
    img = Image.new("RGB", size, color=color)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

mock_ds = datasets.Dataset.from_dict(
    {
        "image": [make_png_bytes("red"), make_png_bytes("blue"), make_png_bytes("green")],
        "label": [0, 1, 2],
    }
)

parquet_path = "mock_images.parquet"
mock_ds.to_parquet(parquet_path)

# 2) Load local parquet with datasets
local_ds = datasets.load_dataset("parquet", data_files={"train": parquet_path})

# 3) Tell datasets that "image" is an image column (decode to PIL on access)
local_ds = local_ds.cast_column("image", datasets.Image())

print(local_ds)
example = local_ds["train"][0]
print(type(example["image"]), example["image"].size, example["label"])
display(example["image"]) # Displays the image directly in the notebook

In [ ]:
import os
import tempfile
import fiftyone as fo

img_dir = tempfile.mkdtemp(prefix="fo_images_")

fo_dataset_name = "hf_local_ds_demo"
if fo_dataset_name in fo.list_datasets():
    fo.delete_dataset(fo_dataset_name)

fo_dataset = fo.Dataset(fo_dataset_name)

for split, hf_split in local_ds.items():
    samples = []

    for i, item in enumerate(hf_split):
        image_path = os.path.join(img_dir, f"{split}_{i:05d}.png")
        item["image"].save(image_path)

        fields = {k: v for k, v in item.items() if k != "image"}
        if "label" in fields:
            fields["ground_truth"] = fo.Classification(label=str(fields.pop("label")))

        samples.append(
            fo.Sample(
                filepath=image_path,
                tags=[split],
                **fields,
            )
        )

    fo_dataset.add_samples(samples)

session = fo.launch_app(fo_dataset)
session

In [ ]:
# Export the existing FiftyOne dataset to a parquet file that `datasets` can read

export_rows = []
for sample in fo_dataset:
	with open(sample.filepath, "rb") as f:
		image_bytes = f.read()

	label = None
	if sample.ground_truth is not None:
		label = sample.ground_truth.label
		if isinstance(label, str) and label.isdigit():
			label = int(label)

	export_rows.append(
		{
			"image": image_bytes,
			"label": label,
		}
	)

hf_from_fo = datasets.Dataset.from_list(export_rows)

fo_parquet_path = os.path.join(img_dir, f"{fo_dataset_name}_export.parquet")
hf_from_fo.to_parquet(fo_parquet_path)

# Load it back with `datasets`
reloaded_ds = datasets.load_dataset("parquet", data_files={"train": fo_parquet_path})
reloaded_ds = reloaded_ds.cast_column("image", datasets.Image())

print("Parquet written to:", fo_parquet_path)
print(reloaded_ds)

reloaded_example = reloaded_ds["train"][0]
print(type(reloaded_example["image"]), reloaded_example["image"].size, reloaded_example["label"])
display(reloaded_example["image"])